In [1]:
import pandas as pd
import numpy as np

stats = pd.read_csv("../data/epl_player_stats_24_25.csv")

# fix percentage columns (text -> numbers)
pct_cols = [c for c in stats.columns if "%" in c]
for c in pct_cols:
    stats[c] = stats[c].astype(str).str.replace("%", "", regex=False).astype(float)

# fix club name, drop the two broken columns
stats["Club"] = stats["Club"].replace({"Brighton": "Brighton & Hove Albion"})
stats = stats.drop(columns=["Shots On Target", "Conversion %"])

# fix known position mislabels
stats.loc[stats["Player Name"] == "Emerson", "Position"] = "DEF"
stats.loc[stats["Player Name"] == "Alex Palmer", "Position"] = "GKP"  # confirm with the saves check

# keep outfield players with 900+ minutes and a populated stats block
df = stats[(stats["Position"] != "GKP")
           & (stats["Minutes"] >= 900)
           & (stats["Passes"] > 0)].copy()

# drop columns with no information for outfield players
num_cols = df.select_dtypes("number").columns.drop(["Appearances", "Minutes"])
useless = [c for c in num_cols if df[c].nunique() <= 1]
df = df.drop(columns=useless)

# convert counting stats to per-90 rates
id_cols = ["Player Name", "Club", "Nationality", "Position"]
pct_cols = [c for c in df.columns if "%" in c]
count_cols = [c for c in df.columns if c not in id_cols + pct_cols + ["Appearances", "Minutes"]]
per90 = df[count_cols].div(df["Minutes"], axis=0) * 90
per90.columns = [c + "_p90" for c in count_cols]
df = pd.concat([df[id_cols + ["Appearances", "Minutes"] + pct_cols], per90], axis=1)

df.to_csv("../data/stats_clean.csv", index=False)
print(df.shape, "| dropped:", useless)

(295, 47) | dropped: ['Goals Conceded', 'xGoT Conceded', 'Saves', 'Saves %', 'Penalties Saved', 'Punches', 'High Claims', 'Goals Prevented']


In [2]:
import pandas as pd

tm_players = pd.read_csv("../data/players.csv")
vals = pd.read_csv("../data/player_valuations.csv")

print(tm_players.shape)
print(tm_players.columns.tolist())
print(vals.shape)
print(vals.columns.tolist())

vals["date"] = pd.to_datetime(vals["date"])
print(vals["date"].min(), vals["date"].max())
print(vals[(vals["date"] >= "2025-04-01") & (vals["date"] <= "2025-07-31")].shape)

FileNotFoundError: [Errno 2] No such file or directory: '../data/players.csv'

In [3]:
import os
print(os.listdir("../data"))


['stats_clean.csv', 'epl_player_stats_24_25.csv', 'player_valuations.csv', 'players.csv', '.ipynb_checkpoints']


In [4]:
import pandas as pd

tm_players = pd.read_csv("../data/players.csv")
vals = pd.read_csv("../data/player_valuations.csv")

print(tm_players.shape)
print(tm_players.columns.tolist())
print(vals.shape)
print(vals.columns.tolist())

vals["date"] = pd.to_datetime(vals["date"])
print(vals["date"].min(), vals["date"].max())
print(vals[(vals["date"] >= "2025-04-01") & (vals["date"] <= "2025-07-31")].shape)

(50149, 26)
['player_id', 'first_name', 'last_name', 'name', 'last_season', 'current_club_id', 'player_code', 'country_of_birth', 'city_of_birth', 'country_of_citizenship', 'date_of_birth', 'sub_position', 'position', 'foot', 'height_in_cm', 'contract_expiration_date', 'agent_name', 'image_url', 'international_caps', 'international_goals', 'current_national_team_id', 'url', 'current_club_domestic_competition_id', 'current_club_name', 'market_value_in_eur', 'highest_market_value_in_eur']
(656301, 6)
['player_id', 'date', 'market_value_in_eur', 'current_club_name', 'current_club_id', 'player_club_domestic_competition_id']
2000-01-20 00:00:00 2026-06-12 00:00:00
(18838, 6)


In [5]:
print(vals["player_club_domestic_competition_id"].value_counts().head(10))

player_club_domestic_competition_id
IT1    50413
TR1    47666
ES1    37561
L1     37049
GB1    33748
FR1    33560
GR1    33498
RU1    32415
PO1    30930
NL1    29860
Name: count, dtype: int64


In [6]:
cutoff = pd.Timestamp("2025-06-30")

snap = (vals[vals["date"] <= cutoff]
        .sort_values("date")
        .groupby("player_id", as_index=False)
        .last())

snap = snap[snap["player_club_domestic_competition_id"] == "GB1"].copy()
snap["days_old"] = (cutoff - snap["date"]).dt.days

print(snap.shape)
print(snap["days_old"].describe())
print(snap["current_club_name"].value_counts())

(1901, 7)
count    1901.000000
mean      879.990005
std      1120.229380
min         0.000000
25%        31.000000
50%       614.000000
75%       970.000000
max      5449.000000
Name: days_old, dtype: float64
current_club_name
Without Club             54
Southampton FC           39
AFC Bournemouth          39
Chelsea FC               39
Crystal Palace           39
                         ..
Manchester United U21     1
Brentford FC B            1
Lancaster City FC         1
Paris Saint-Germain       1
Juventus Next Gen         1
Name: count, Length: 365, dtype: int64


In [7]:
snap = snap[snap["days_old"] <= 365].copy()

snap = snap.merge(
    tm_players[["player_id", "name", "date_of_birth", "position", "sub_position"]],
    on="player_id", how="left")

snap["age"] = ((pd.Timestamp("2025-05-25") - pd.to_datetime(snap["date_of_birth"])).dt.days / 365.25).round(1)

print(snap.shape)
snap[["name", "current_club_name", "market_value_in_eur", "age", "days_old"]].sort_values(
    "market_value_in_eur", ascending=False).head(10)

(783, 12)


,name,current_club_name,market_value_in_eur,age,days_old
377,Erling Haaland,Manchester City,180000000,24.8,31
396,Bukayo Saka,Arsenal FC,150000000,23.7,31
288,Alexander Isak,Newcastle United,120000000,25.7,31
537,Cole Palmer,Chelsea FC,120000000,23.1,31
302,Declan Rice,Arsenal FC,120000000,26.4,31
299,Rodri,Manchester City,110000000,28.9,31
356,Phil Foden,Manchester City,100000000,25.0,31
505,Alexis Mac Allister,Liverpool FC,100000000,26.4,31
638,Moisés Caicedo,Chelsea FC,90000000,23.6,31
234,Martin Ødegaard,Arsenal FC,85000000,26.4,31


In [8]:
import unicodedata, re

def norm(s):
    s = str(s)
    for a, b in {"ø": "o", "Ø": "O", "ß": "ss", "đ": "d", "ł": "l", "æ": "ae"}.items():
        s = s.replace(a, b)
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode()
    s = re.sub(r"[^a-z0-9 ]", "", s.lower().replace("-", " "))
    return re.sub(r"\s+", " ", s).strip()

def norm_club(s):
    s = re.sub(r"\b(afc|fc)\b", "", norm(s))
    return re.sub(r"\s+", " ", s).strip()

df = pd.read_csv("../data/stats_clean.csv")
df["name_key"] = df["Player Name"].apply(norm)
df["club_key"] = df["Club"].apply(norm_club)

snap["name_key"] = snap["name"].apply(norm)
snap["club_key"] = snap["current_club_name"].apply(norm_club)

# our clubs that have no equivalent in the Transfermarkt table
print(sorted(set(df["club_key"]) - set(snap["club_key"])))

[]


In [9]:
m = df.merge(
    snap[["name_key", "club_key", "market_value_in_eur", "age", "days_old", "date_of_birth"]],
    on=["name_key", "club_key"], how="left")

print("rows:", len(m), "(should still be", len(df), ")")
print("matched:", m["market_value_in_eur"].notna().sum())

rows: 295 (should still be 295 )
matched: 278


In [10]:
m[m["market_value_in_eur"].isna()][["Player Name", "Club", "Position", "Minutes"]].sort_values("Minutes", ascending=False)

,Player Name,Club,Position,Minutes
36,Illia Zabarnyi,Bournemouth,DEF,3113
118,Vitalii Mykolenko,Everton,DEF,3084
294,Toti Gomes,Wolverhampton Wanderers,DEF,2615
165,Andy Robertson,Liverpool,DEF,2492
3,Gabriel Magalhães,Arsenal,DEF,2365
150,Boubakary Soumaré,Leicester City,MID,2170
264,Son Heung-Min,Tottenham Hotspur,MID,2116
260,Pape Sarr,Tottenham Hotspur,MID,1913
97,Jordan Ayew,Crystal Palace,MID,1664
55,Yehor Yarmoliuk,Brentford,MID,1442


In [11]:
unmatched = m[m["market_value_in_eur"].isna()]

for _, r in unmatched.iterrows():
    tokens = [t for t in r["name_key"].split() if len(t) > 3]
    mask = snap["name_key"].apply(lambda s: any(t in s for t in tokens))
    cand = snap[mask][["name", "current_club_name", "market_value_in_eur", "age", "days_old"]]
    print(f"{r['Player Name']} | {r['Club']}")
    print(cand.head(8).to_string(index=False) if len(cand) else "  no candidates in snapshot")
    print()

Gabriel Magalhães | Arsenal
              name current_club_name  market_value_in_eur  age  days_old
     Gabriel Jesus        Arsenal FC             32000000 28.1        31
           Gabriel        Arsenal FC             75000000 27.4        31
Gabriel Martinelli        Arsenal FC             55000000 23.9        31
   Gabriel Slonina        Chelsea FC              3500000 21.0        31

Illia Zabarnyi | Bournemouth
          name current_club_name  market_value_in_eur  age  days_old
       Willian         Fulham FC               750000 36.8        31
William Saliba        Arsenal FC             80000000 24.2        31
 Rhys Williams      Liverpool FC               700000 24.3         0
 Neco Williams Nottingham Forest             20000000 24.1        31
 William Osula  Newcastle United              7000000 21.8        31
 Ilya Zabarnyi   AFC Bournemouth             42000000 22.7        31

Mads Roerslev | Brentford
          name   current_club_name  market_value_in_eur  age  days_

In [12]:
manual = {
    "Gabriel Magalhães": "Gabriel",
    "Illia Zabarnyi": "Ilya Zabarnyi",
    "Vitalii Mykolenko": "Vitaliy Mykolenko",
    "Sam Szmodics": "Sammie Szmodics",
    "Andy Robertson": "Andrew Robertson",
    "Pape Sarr": "Pape Matar Sarr",
    "Son Heung-Min": "Heung-min Son",
    "Toti Gomes": "Toti",
    "Jordan Ayew": "Jordan Ayew",
    "Adam Armstrong": "Adam Armstrong",
}

fill_cols = ["market_value_in_eur", "age", "days_old", "date_of_birth"]
m["match_method"] = np.where(m["market_value_in_eur"].notna(), "exact", "none")

for fb_name, tm_name in manual.items():
    rows = snap[snap["name"] == tm_name]
    if len(rows) != 1:
        print("CHECK:", fb_name, "->", tm_name, "found", len(rows), "rows")
        continue
    idx = m.index[m["Player Name"] == fb_name]
    for c in fill_cols:
        m.loc[idx, c] = rows.iloc[0][c]
    m.loc[idx, "match_method"] = "manual"

print(m["match_method"].value_counts())
m[m["match_method"] == "manual"][["Player Name", "Club", "market_value_in_eur", "age"]]

match_method
exact     278
manual     10
none        7
Name: count, dtype: int64


,Player Name,Club,market_value_in_eur,age
3,Gabriel Magalhães,Arsenal,75000000.0,27.4
36,Illia Zabarnyi,Bournemouth,42000000.0,22.7
97,Jordan Ayew,Crystal Palace,2500000.0,33.7
103,Adam Armstrong,Everton,12000000.0,28.3
118,Vitalii Mykolenko,Everton,28000000.0,26.0
148,Sam Szmodics,Ipswich Town,10000000.0,29.7
165,Andy Robertson,Liverpool,18000000.0,31.2
260,Pape Sarr,Tottenham Hotspur,32000000.0,22.7
264,Son Heung-Min,Tottenham Hotspur,20000000.0,32.9
294,Toti Gomes,Wolverhampton Wanderers,28000000.0,26.4


In [13]:
terms = ["slev", "yarmol", "soumar", "okoli", "wes(?:ley)? burns", "lemina", "kyle walker"]

for t in terms:
    found = tm_players[tm_players["name"].apply(norm).str.contains(t)][
        ["player_id", "name", "date_of_birth", "current_club_name"]]
    last = (vals[vals["player_id"].isin(found["player_id"]) & (vals["date"] <= cutoff)]
            .sort_values("date").groupby("player_id").last()
            [["date", "market_value_in_eur", "current_club_name"]].reset_index())
    print("---", t)
    print(found.merge(last, on="player_id", how="left", suffixes=("", "_val")).head(10).to_string(index=False))

--- slev
 player_id              name       date_of_birth   current_club_name       date  market_value_in_eur current_club_name_val
    391005     Mads Roerslev 1999-06-24 00:00:00        Brentford FC 2025-06-05              6000000         VfL Wolfsburg
    495488 Rasmus Vinderslev 1997-08-12 00:00:00 Sönderjyske Fodbold 2025-06-16               800000   Sönderjyske Fodbold
--- yarmol
 player_id               name       date_of_birth  current_club_name       date  market_value_in_eur       current_club_name_val
     69015  Andriy Yarmolenko 1989-10-23 00:00:00        Dynamo Kyiv 2025-06-10              1500000                 Dynamo Kyiv
    260315 Artem Yarmolitskiy 1994-04-10 00:00:00       Arsenal Tula 2019-06-14               200000 Khimik-Arsenal Novomoskovsk
    396524   Artem Yarmolenko 1998-01-18 00:00:00 Chornomorets Odesa 2021-06-17                25000        Agrobiznes Volochysk
    717411    Yegor Yarmolyuk 2004-03-01 00:00:00       Brentford FC 2025-05-30             150

In [14]:
manual_ids = {
    "Mads Roerslev": 391005,
    "Yehor Yarmoliuk": 717411,
    "Mario Lemina": 170934,
    "Kyle Walker": 95424,
}
season_end = pd.Timestamp("2025-05-25")

for fb_name, pid in manual_ids.items():
    v = (vals[(vals["player_id"] == pid) & (vals["date"] <= cutoff)]
         .sort_values("date").iloc[-1])
    dob = pd.to_datetime(tm_players.loc[tm_players["player_id"] == pid, "date_of_birth"].iloc[0])
    idx = m.index[m["Player Name"] == fb_name]
    assert len(idx) == 1, fb_name
    m.loc[idx, "market_value_in_eur"] = v["market_value_in_eur"]
    m.loc[idx, "days_old"] = (cutoff - v["date"]).days
    m.loc[idx, "age"] = round((season_end - dob).days / 365.25, 1)
    m.loc[idx, "date_of_birth"] = dob
    m.loc[idx, "match_method"] = "manual"
    print(fb_name, "|", v["current_club_name"], "|", v["date"].date(), "|", v["market_value_in_eur"])

# drop the three with stale values
drop_names = ["Boubakary Soumaré", "Caleb Okoli", "Wes Burns"]
final = m[~m["Player Name"].isin(drop_names)].copy()
assert final["market_value_in_eur"].notna().all()

final["club_mismatch"] = final["Player Name"].isin(
    ["Jordan Ayew", "Adam Armstrong", "Mads Roerslev", "Mario Lemina", "Kyle Walker"])

final.to_csv("../data/model_data.csv", index=False)
print(final.shape)
print(final[["market_value_in_eur", "age", "days_old"]].describe())

TypeError: Invalid value '1999-06-24 00:00:00' for dtype 'str'. Value should be a string or missing value, got 'Timestamp' instead.

In [15]:
manual_ids = {
    "Mads Roerslev": 391005,
    "Yehor Yarmoliuk": 717411,
    "Mario Lemina": 170934,
    "Kyle Walker": 95424,
}
season_end = pd.Timestamp("2025-05-25")

for fb_name, pid in manual_ids.items():
    v = (vals[(vals["player_id"] == pid) & (vals["date"] <= cutoff)]
         .sort_values("date").iloc[-1])
    dob = pd.to_datetime(tm_players.loc[tm_players["player_id"] == pid, "date_of_birth"].iloc[0])
    idx = m.index[m["Player Name"] == fb_name]
    assert len(idx) == 1, fb_name
    m.loc[idx, "market_value_in_eur"] = v["market_value_in_eur"]
    m.loc[idx, "days_old"] = (cutoff - v["date"]).days
    m.loc[idx, "age"] = round((season_end - dob).days / 365.25, 1)
    m.loc[idx, "date_of_birth"] = dob
    m.loc[idx, "match_method"] = "manual"
    print(fb_name, "|", v["current_club_name"], "|", v["date"].date(), "|", v["market_value_in_eur"])

# drop the three with stale values
drop_names = ["Boubakary Soumaré", "Caleb Okoli", "Wes Burns"]
final = m[~m["Player Name"].isin(drop_names)].copy()
assert final["market_value_in_eur"].notna().all()

final["club_mismatch"] = final["Player Name"].isin(
    ["Jordan Ayew", "Adam Armstrong", "Mads Roerslev", "Mario Lemina", "Kyle Walker"])

final.to_csv("../data/model_data.csv", index=False)
print(final.shape)
print(final[["market_value_in_eur", "age", "days_old"]].describe())

TypeError: Invalid value '1999-06-24 00:00:00' for dtype 'str'. Value should be a string or missing value, got 'Timestamp' instead.

In [16]:
manual_ids = {
    "Mads Roerslev": 391005,
    "Yehor Yarmoliuk": 717411,
    "Mario Lemina": 170934,
    "Kyle Walker": 95424,
}
season_end = pd.Timestamp("2025-05-25")

m["date_of_birth"] = pd.to_datetime(m["date_of_birth"])   # <- new line

for fb_name, pid in manual_ids.items():
    v = (vals[(vals["player_id"] == pid) & (vals["date"] <= cutoff)]
         .sort_values("date").iloc[-1])
    dob = pd.to_datetime(tm_players.loc[tm_players["player_id"] == pid, "date_of_birth"].iloc[0])
    idx = m.index[m["Player Name"] == fb_name]
    assert len(idx) == 1, fb_name
    m.loc[idx, "market_value_in_eur"] = v["market_value_in_eur"]
    m.loc[idx, "days_old"] = (cutoff - v["date"]).days
    m.loc[idx, "age"] = round((season_end - dob).days / 365.25, 1)
    m.loc[idx, "date_of_birth"] = dob
    m.loc[idx, "match_method"] = "manual"
    print(fb_name, "|", v["current_club_name"], "|", v["date"].date(), "|", v["market_value_in_eur"])

# drop the three with stale values
drop_names = ["Boubakary Soumaré", "Caleb Okoli", "Wes Burns"]
final = m[~m["Player Name"].isin(drop_names)].copy()
assert final["market_value_in_eur"].notna().all()

final["club_mismatch"] = final["Player Name"].isin(
    ["Jordan Ayew", "Adam Armstrong", "Mads Roerslev", "Mario Lemina", "Kyle Walker"])

final.to_csv("../data/model_data.csv", index=False)
print(final.shape)
print(final[["market_value_in_eur", "age", "days_old"]].describe())

Mads Roerslev | VfL Wolfsburg | 2025-06-05 | 6000000
Yehor Yarmoliuk | Brentford FC | 2025-05-30 | 15000000
Mario Lemina | Galatasaray | 2025-06-18 | 7500000
Kyle Walker | AC Milan | 2025-06-17 | 4000000
(292, 55)
       market_value_in_eur         age    days_old
count         2.920000e+02  292.000000  292.000000
mean          2.999640e+07   26.820890   30.863014
std           2.435324e+07    3.761846    1.577898
min           5.000000e+05   18.700000   12.000000
25%           1.375000e+07   24.100000   31.000000
50%           2.500000e+07   26.450000   31.000000
75%           4.000000e+07   28.825000   31.000000
max           1.800000e+08   39.900000   34.000000


In [17]:
cols = ["Player Name", "Club", "Position", "age", "market_value_in_eur"]
print(final.sort_values("market_value_in_eur", ascending=False)[cols].head(10))
print(final.sort_values("market_value_in_eur")[cols].head(10))

             Player Name              Club Position   age  market_value_in_eur
178       Erling Haaland   Manchester City      FWD  24.8          180000000.0
1            Bukayo Saka           Arsenal      MID  23.7          150000000.0
2            Declan Rice           Arsenal      MID  26.4          120000000.0
74           Cole Palmer           Chelsea      MID  23.1          120000000.0
208       Alexander Isak  Newcastle United      FWD  25.7          120000000.0
188           Phil Foden   Manchester City      MID  25.0          100000000.0
164  Alexis Mac Allister         Liverpool      MID  26.4          100000000.0
80        Moisés Caicedo           Chelsea      MID  23.6           90000000.0
9        Martin Ødegaard           Arsenal      MID  26.4           85000000.0
210      Bruno Guimarães  Newcastle United      MID  27.5           80000000.0
            Player Name                     Club Position   age  \
104        Ashley Young                  Everton      DEF  39.9 